# 🎬 SadTalker Studio SE — Talande Avatar

**Ladda upp ett foto + skriv text på svenska → Få en talande video**

Baserat på [SadTalker](https://github.com/OpenTalker/SadTalker) av OpenTalker (MIT-licens).

---

## ⚡ GPU-krav

| GPU | VRAM | Status |
|-----|------|--------|
| **T4 (15 GB)** | 15 GB | ✅ Fungerar — **gratis Colab** |
| **V100 (16 GB)** | 16 GB | ✅ Fungerar |
| **A100 (40 GB)** | 40 GB | ✅ Fungerar |
| CPU | — | ❌ För långsamt |

> **Aktivera GPU:** Runtime → Change runtime type → **T4 GPU** *(gratis)*

---

## ⏱️ Tidsuppskattning

- **Första sessionen:** ~10 min (installation + ~1.3 GB modeller laddas ner)
- **Efterföljande sessioner:** ~2 min setup (modeller hämtas från Drive)
- **Per video (256 px):** ~1–2 min
- **Per video (512 px):** ~3–4 min

---

## 🔒 GDPR / Dataskydd

- Foto och ljud bearbetas i din Colab-VM och rensas när sessionen avslutas
- Modeller (~1.3 GB) sparas i **ditt** Google Drive — ditt eget konto
- **OBS:** TTS-texten skickas till **Microsoft Edge TTS API** (speech.platform.bing.com) — inkludera inte känsliga personuppgifter i texten

In [ ]:
# Cell 2 — Kontrollera GPU
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_line = result.stdout.strip()
print(f'GPU: {gpu_line}')

try:
    vram_mb = int(gpu_line.split(',')[1].strip().split()[0])
    vram_gb = vram_mb / 1024
    if vram_gb < 3:
        print(f'❌ VARNING: {vram_gb:.1f} GB VRAM — SadTalker behöver minst 4 GB.')
        print('   Aktivera GPU: Runtime → Change runtime type → T4 GPU')
    else:
        print(f'✅ {vram_gb:.1f} GB VRAM — Bra, SadTalker fungerar!')
except Exception:
    print('ℹ️  Ingen GPU hittad — aktivera GPU under Runtime → Change runtime type')

In [ ]:
# Cell 3 — Montera Google Drive (för att cacha modeller)
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive monterat')

In [ ]:
# Cell 4 — Installera SadTalker (~3-5 min första gången)
import subprocess, sys, os

SADTALKER_DIR = '/content/SadTalker'

if not os.path.exists(SADTALKER_DIR):
    print('📥 Klonar SadTalker...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/OpenTalker/SadTalker.git', SADTALKER_DIR],
        check=True, capture_output=True
    )
    print('✅ Klonat!')
else:
    print('✅ SadTalker redan klonat')

print('📦 Installerar beroenden (~3-5 min)...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    cwd=SADTALKER_DIR, check=True
)
print('✅ Beroenden installerade!')

In [ ]:
# Cell 5 — Ladda ner modeller till Google Drive (en gång, ~1.3 GB + ~350 MB GFPGAN)
import os
import urllib.request
from pathlib import Path
from huggingface_hub import snapshot_download

DRIVE_MODELS   = '/content/drive/MyDrive/SadTalker-Models'
CHECKPOINT_DIR = f'{DRIVE_MODELS}/checkpoints'
GFPGAN_DIR     = f'{DRIVE_MODELS}/gfpgan/weights'
SADTALKER_DIR  = '/content/SadTalker'
MARKER         = f'{DRIVE_MODELS}/.download_complete'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(GFPGAN_DIR, exist_ok=True)

if os.path.exists(MARKER):
    print('✅ Modeller redan cachade i Drive — hoppar över nedladdning')
else:
    print('📥 Laddar ner SadTalker-modeller (~1.3 GB)...')
    print('   Görs bara EN gång — sedan cachas allt i Drive.')
    snapshot_download(
        repo_id='vinthony/SadTalker',
        local_dir=CHECKPOINT_DIR,
        ignore_patterns=['*.md', '*.txt', '.git*']
    )
    print('✅ SadTalker-modeller nedladdade!')

    gfpgan_path = f'{GFPGAN_DIR}/GFPGANv1.4.pth'
    if not os.path.exists(gfpgan_path):
        print('📥 Laddar ner GFPGAN ansiktsförbättrare (~350 MB)...')
        urllib.request.urlretrieve(
            'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth',
            gfpgan_path
        )
        print('✅ GFPGAN nedladdad!')

    Path(MARKER).touch()
    print('✅ Alla modeller klara och cachade!')

# Skapa symlinks till SadTalkers förväntade sökvägar
ck_link = f'{SADTALKER_DIR}/checkpoints'
if not os.path.islink(ck_link) and not os.path.exists(ck_link):
    os.symlink(CHECKPOINT_DIR, ck_link)
    print(f'✅ checkpoints → Drive')
else:
    print('✅ checkpoints-symlink OK')

gfpgan_weights_link = f'{SADTALKER_DIR}/gfpgan/weights'
os.makedirs(f'{SADTALKER_DIR}/gfpgan', exist_ok=True)
if not os.path.islink(gfpgan_weights_link) and not os.path.exists(gfpgan_weights_link):
    os.symlink(GFPGAN_DIR, gfpgan_weights_link)
    print('✅ gfpgan/weights → Drive')
else:
    print('✅ gfpgan/weights-symlink OK')

In [ ]:
# Cell 6 — Installera edge-tts + verifiera allt
import subprocess, sys, os

SADTALKER_DIR  = '/content/SadTalker'
CHECKPOINT_DIR = '/content/drive/MyDrive/SadTalker-Models/checkpoints'
GFPGAN_PATH    = '/content/drive/MyDrive/SadTalker-Models/gfpgan/weights/GFPGANv1.4.pth'

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'edge-tts', '-q'],
    check=True
)
print('✅ edge-tts installerat')

checks = [
    (f'{SADTALKER_DIR}/inference.py',                           'SadTalker inference-skript'),
    (f'{SADTALKER_DIR}/checkpoints',                            'Checkpoints-symlink'),
    (f'{CHECKPOINT_DIR}/SadTalker_V0.0.2_256.safetensors',     'Modell 256px'),
    (f'{CHECKPOINT_DIR}/SadTalker_V0.0.2_512.safetensors',     'Modell 512px'),
    (f'{CHECKPOINT_DIR}/auido2exp_00300-model.pth',             'Audio→Uttryck'),
    (f'{CHECKPOINT_DIR}/auido2pose_00140-model.pth',            'Audio→Pose'),
    (f'{CHECKPOINT_DIR}/facevid2vid_00189-model.pth.tar',       'FaceVid2Vid'),
    (GFPGAN_PATH,                                               'GFPGAN ansiktsförbättrare'),
]

all_ok = True
for path, label in checks:
    exists = os.path.exists(path)
    print(f'{'✅' if exists else '❌'} {label}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('🚀 Allt klart! Kör Cell 7 för att starta Gradio-gränssnittet.')
else:
    print('⚠️  Saknade filer — kör Cell 5 igen.')

In [ ]:
# Cell 7 — 🎬 Starta Gradio UI  ← KÖR DETTA FÖR ATT ANVÄNDA
import gradio as gr
import edge_tts
import asyncio
import subprocess
import sys
import os
import shutil
from datetime import datetime
from pathlib import Path

SADTALKER_DIR  = '/content/SadTalker'
CHECKPOINT_DIR = '/content/drive/MyDrive/SadTalker-Models/checkpoints'
OUTPUTS_DIR    = '/content/sadtalker_outputs'
TTS_MAX_CHARS  = 2000

os.makedirs(OUTPUTS_DIR, exist_ok=True)

SWEDISH_VOICES = {
    'Sofie (kvinna, rikssvenska)':     'sv-SE-SofieNeural',
    'Mattias (man, rikssvenska)':      'sv-SE-MattiasNeural',
    'Noora (kvinna, finlandssvenska)': 'sv-FI-NooraNeural',
}

async def _tts_async(text, voice, out_wav):
    communicate = edge_tts.Communicate(text, voice)
    mp3_path = out_wav.replace('.wav', '.mp3')
    await communicate.save(mp3_path)
    subprocess.run(
        ['ffmpeg', '-y', '-i', mp3_path, '-ar', '16000', '-ac', '1', out_wav],
        check=True, capture_output=True
    )
    os.remove(mp3_path)

def _run_tts(text, voice, out_wav):
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(_tts_async(text, voice, out_wav))
    finally:
        loop.close()

def generate_avatar(photo_path, text, voice_name, still_mode, size, use_enhancer):
    if photo_path is None:
        return None, '❌ Ladda upp ett foto först'
    if not text.strip():
        return None, '❌ Skriv in texten som personen ska säga'
    if len(text) > TTS_MAX_CHARS:
        return None, f'❌ Texten är för lång ({len(text)} tecken, max {TTS_MAX_CHARS})'

    ts = datetime.now().strftime('%H%M%S')
    tmp_dir = f'/tmp/sadtalker_{ts}'
    os.makedirs(tmp_dir, exist_ok=True)

    try:
        # TTS: text → MP3 → WAV 16kHz mono
        audio_path = f'{tmp_dir}/audio.wav'
        _run_tts(text, SWEDISH_VOICES[voice_name], audio_path)

        if not os.path.exists(audio_path):
            return None, '❌ TTS misslyckades — kontrollera internetanslutningen'

        # SadTalker inference
        out_dir = f'{tmp_dir}/output'
        os.makedirs(out_dir, exist_ok=True)

        cmd = [
            sys.executable, 'inference.py',
            '--driven_audio',   audio_path,
            '--source_image',   photo_path,
            '--checkpoint_dir', CHECKPOINT_DIR,
            '--result_dir',     out_dir,
            '--preprocess',     'full',
            '--size',           str(size),
        ]
        if still_mode:
            cmd.append('--still')
        if use_enhancer:
            cmd += ['--enhancer', 'gfpgan']

        result = subprocess.run(
            cmd, cwd=SADTALKER_DIR,
            timeout=600, capture_output=True, text=True
        )

        if result.returncode != 0:
            err = (result.stderr or result.stdout or 'okänt fel')[-800:]
            return None, f'❌ Inference misslyckades:\n{err}'

        videos = list(Path(out_dir).rglob('*.mp4'))
        if not videos:
            return None, '❌ Ingen video hittades — se felmeddelande ovan'

        best = max(videos, key=lambda p: p.stat().st_size)
        final_path = os.path.join(OUTPUTS_DIR, f'video_{ts}.mp4')
        shutil.copy(str(best), final_path)

        size_kb = best.stat().st_size // 1024
        return final_path, f'✅ Klar! video_{ts}.mp4 ({size_kb} KB)'

    except subprocess.TimeoutExpired:
        return None, '❌ Timeout (10 min) — prova kortare text eller storlek 256'
    except Exception as e:
        return None, f'❌ Oväntat fel: {e}'
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


with gr.Blocks(title='SadTalker Studio SE', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
# 🎬 SadTalker Studio SE
**Ladda upp ett foto + skriv svenska → Talande video på ~1–4 min**  
*TTS: Microsoft Edge (gratis, ingen API-nyckel) · Video: SadTalker (MIT-licens, open source)*
""")

    with gr.Row():
        with gr.Column(scale=1):
            photo = gr.Image(
                label='📷 Foto (porträttbild med tydligt ansikte fungerar bäst)',
                type='filepath'
            )
            voice = gr.Dropdown(
                choices=list(SWEDISH_VOICES.keys()),
                value='Sofie (kvinna, rikssvenska)',
                label='🎤 Röst'
            )
            text = gr.Textbox(
                label='📝 Text att tala (max 2 000 tecken)',
                placeholder='Hej! Välkommen till vår butik. Vi erbjuder...',
                lines=6
            )
            with gr.Row():
                still   = gr.Checkbox(label='Stilla (minimala huvudrörelser)', value=True)
                enhance = gr.Checkbox(label='Förbättra ansikte / GFPGAN (tar längre)', value=False)
            size = gr.Radio(
                choices=[256, 512], value=256,
                label='📐 Kvalitet  (256 = ~1–2 min · 512 = ~3–4 min)'
            )
            btn = gr.Button('🎬 Generera talande video', variant='primary', size='lg')

        with gr.Column(scale=1):
            video_out = gr.Video(label='🎬 Resultat')
            status    = gr.Textbox(label='Status', interactive=False, lines=4)
            gr.Markdown("""
**Tips för bäst resultat:**
- Porträttfoto med tydligt ansikte + neutral bakgrund
- Välbelyst och skarp bild (JPG eller PNG)
- Välj *Stilla* för professionella business-videor
- Aktivera *Förbättra ansikte* för skarpare slutresultat (tar ~1 min extra)

**Sparade videor:** `/content/sadtalker_outputs/`
""")

    btn.click(
        fn=generate_avatar,
        inputs=[photo, text, voice, still, size, enhance],
        outputs=[video_out, status]
    )

demo.launch(share=True, debug=False, show_error=True)
print('\n🔗 Kopiera gradio.live-länken ovan → klistra in i dashboarden → klicka Anslut')